# QuEstX: Graph Transformer Training and Evaluation

This notebook defines the neural architecture and executes the supervised regression task to predict the Probability of Successful Trial (PST).

1. Dependency Installation

In [ ]:
# Install essential libraries for GNNs, quantum simulation, and hardware metrics
%pip install torch-geometric
%pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-$(torch.__version__).html

import torch
import torch.nn.functional as F
from torch.nn import Linear, ReLU, Sequential
from torch_geometric.nn import TransformerConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import pearsonr
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

2. Model Architecture (QuEst Structure)

This implementation follows the Graph Transformer architecture described in Section 4.3 of the article.

In [ ]:
class QuEstTransformer(torch.nn.Module):
    """
    Implementation of the Graph Transformer proposed in the QuEst paper.
    The architecture follows:
    1. Node feature input
    2. Multiple Graph Transformer layers (PyG TransformerConv)
    3. Global Average Pooling (to obtain a single vector per graph)
    4. An MLP regressor to predict the final PST value.
    """
    def __init__(self, in_channels, hidden_channels, global_feature_dim,
             out_channels=1, num_layers=2, heads=1, dropout=0.0):
        super(QuEstTransformer, self).__init__()

        # Graph Transformer Layers (TransformerConv allows global attention across gates)
        self.transformer_layers = torch.nn.ModuleList()
        for i in range(num_layers):
            in_dim = in_channels if i == 0 else hidden_channels
            conv = TransformerConv(in_dim,
                               hidden_channels,
                               heads=heads,       # Single-head as per the original paper
                               dropout=dropout,
                               concat=False)
            self.transformer_layers.append(conv)

        self.pool = global_mean_pool

        # Global Features Pre-processor (2 FC layers, dim=12)
        # Based on Section 5.1: "two FC layers with hidden and output dimensions of 12"
        self.global_fc = Sequential(
            Linear(global_feature_dim, 12),
            ReLU(),
            Linear(12, 12)
        )

        # Final Regressor: receives pooled node embeddings + global features
        # 3 FC layers, hidden_dim=128, as per Section 5.1 of QuEst
        self.regressor = Sequential(
            Linear(hidden_channels + 12, 128),
            ReLU(),
            Linear(128, 128),
            ReLU(),
            Linear(128, out_channels)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Propagation through Transformer layers
        for conv in self.transformer_layers:
            x = conv(x, edge_index)
            x = F.relu(x)

        # Global average pooling → graph-level embedding
        x_pooled = self.pool(x, batch)  # shape: [num_graphs, hidden_channels]

        # Process global circuit features (depth, width, etc.)
        gf = self.global_fc(data.global_features)  # shape: [num_graphs, 12]

        # Concatenate embeddings before the regression head
        x_cat = torch.cat([x_pooled, gf], dim=1)  # shape: [num_graphs, hidden_channels + 12]

        return self.regressor(x_cat)

3. Data Loading and Partitioning

The dataset is split into 70% Training, 20% Validation, and 10% Testing, matching the methodology in Section 5.1 of the article.

In [ ]:
# Load normalized PyG data
pickle_filename = 'normalized_data\dataset_D2.data'

print(ff"Loading dataset from '{pickle_filename}'...")
try:
    with open(pickle_filename, 'rb') as f:
        loaded_data = pickle.load(f)
    print("Arquivo pickle carregado com sucesso.")

except FileNotFoundError:
    print(f"ERRO: File '{pickle_filename}' not found.")
    print("Please make sure you uploaded the file and that the name is correct.")

    raise

dataset = loaded_data 

# Reproducible shuffling
torch.manual_seed(42)
np.random.seed(42)

np.random.shuffle(dataset)

# Splits
train_split = int(0.70 * len(dataset))
val_split = int(0.20 * len(dataset))

train_dataset = dataset[:train_split]
val_dataset = dataset[train_split : train_split + val_split]
test_dataset = dataset[train_split + val_split:]

print(f"Total: {len(dataset)} samples")
print(f"Train:    {len(train_dataset)} ({len(train_dataset)/len(dataset):.0%})")
print(f"Validation: {len(val_dataset)}  ({len(val_dataset)/len(dataset):.0%})")
print(f"Test:     {len(test_dataset)}  ({len(test_dataset)/len(dataset):.0%})")


# Move the datasets to the device before creating the DataLoaders.
device = torch.device('cuda')
#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_dataset = [data.to(device) for data in train_dataset]
val_dataset = [data.to(device) for data in val_dataset]
test_dataset = [data.to(device) for data in test_dataset]


# Create DataLoaders
batch_size = 2500
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

4. Training Configuration

In Experiment E3.2, the model capacity was adjusted (Hidden Dimension = 72, Layers = 3) to handle larger qubit scales.

In [ ]:
# Check if the GPU is available.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

# Determine the dimensions of global features from the dataset.
global_feature_dim = train_dataset[0].global_features.shape[1]
print(f"Global feature dimensions: {global_feature_dim}")


input_feature_size = train_dataset[0].x.shape[1] # Dynamically adjusted to the actual size of the dataset
print(f"Input feature dimensions of the node: {input_feature_size}")

# Model Hyperparameters
model = QuEstTransformer(
    in_channels=input_feature_size, # The QuEst article specifies 24 input features per node.
    hidden_channels=72,             # Adjustable
    global_feature_dim=global_feature_dim,
    out_channels=1,                 # Fixed (Regression)
    num_layers=3,                   # Do Artigo
    heads=1,                        # Adjustable
    dropout=0.2                     # Adjustable
).to(device)

print(f"QuEstX model created with {sum(p.numel() for p in model.parameters())} parameters.")

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Loss Function
# Since this is a regression problem (predicting fidelity), we use MSE.
criterion = torch.nn.MSELoss()

Executando no dispositivo: cuda
Dimensão das features globais: 6
Dimensão das features de entrada do nó: 41
Modelo QuEstX criado com 81905 parâmetros.


5. Training and Evaluation Loops

Metrics calculated include RMSE, MAE, Pearson correlation (r), and R2 to match the results reported in Table 5 of the paper.

In [ ]:
def train_epoch(loader):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)

        optimizer.zero_grad()

        out = model(data)

        loss = criterion(out, data.y.to(device).view(-1, 1))

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * data.num_graphs

    return total_loss / len(loader.dataset)

def evaluate_epoch(loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad(): 
        for data in loader:
            data = data.to(device)
            out = model(data)

            all_preds.append(out.cpu())
            all_labels.append(data.y.cpu())

    all_preds_tensor = torch.cat(all_preds)
    all_labels_tensor = torch.cat(all_labels)

    mse = criterion(all_preds_tensor, all_labels_tensor.view(-1, 1)).item()

    all_preds_np = all_preds_tensor.numpy()
    all_labels_np = all_labels_tensor.numpy()

    # 1. RMSE (Root Mean Squared Error)
    rmse = np.sqrt(mse)

    # 2. MAE (Mean Absolute Error)
    mae = mean_absolute_error(all_labels_np, all_preds_np)

    # 3. Pearson (r)
    pearson_r, _ = pearsonr(all_labels_np.squeeze(), all_preds_np.squeeze())

    # 4. R² (R-squared)
    r2 = r2_score(all_labels_np, all_preds_np)

    return {
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'pearson': pearson_r,
        'r2': r2 
    }

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

num_epochs = 500
print("Starting the training...")


history = {
    "epoch": [],
    "train_loss": [],
    "val_mse": [],
    "val_rmse": [],
    "val_mae": [],
    "val_pearson": [],
    "val_r2": [],
}

best_val_mse = float("inf")
best_model_state = None


for epoch in range(1, num_epochs + 1):
    
    train_loss = train_epoch(train_loader)

    
    val_metrics = evaluate_epoch(val_loader)

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_mse"].append(val_metrics["mse"])
    history["val_rmse"].append(val_metrics["rmse"])
    history["val_mae"].append(val_metrics["mae"])
    history["val_pearson"].append(val_metrics["pearson"])
    history["val_r2"].append(val_metrics["r2"])

    if val_metrics["mse"] < best_val_mse:
        best_val_mse = val_metrics["mse"]
        best_model_state = model.state_dict()

    print(
        f'Epoch: {epoch:02d}, '
        f'Train Loss (MSE): {train_loss:.6f}, '
        f'Val Loss (MSE): {val_metrics["mse"]:.6f}, '
        f'Val RMSE: {val_metrics["rmse"]:.6f}, '
        f'Val MAE: {val_metrics["mae"]:.6f}, '
        f'Val Pearson (r): {val_metrics["pearson"]:.4f}, '
        f'Val R²: {val_metrics["r2"]:.4f}'
    )

print("\nTraining completed!")


if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nBest loaded model (Val MSE = {best_val_mse:.6f})")
else:
    print("\nWarning: best_model_state is None; using the last trained model.")


print("\nEvaluating on the test set...")

test_metrics = evaluate_epoch(test_loader)

print(
    "\nMetrics in the TEST set:"
    f"\n  Test MSE     : {test_metrics['mse']:.6f}"
    f"\n  Test RMSE    : {test_metrics['rmse']:.6f}"
    f"\n  Test MAE     : {test_metrics['mae']:.6f}"
    f"\n  Test Pearson : {test_metrics['pearson']:.4f}"
    f"\n  Test R²      : {test_metrics['r2']:.4f}"
)


epochs = history["epoch"]

# 1) Training loss vs Val MSE
plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_loss"], label="Train Loss (MSE)")
plt.plot(epochs, history["val_mse"], label="Val Loss (MSE)")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Training Curve - MSE (Training vs. Validation)")
plt.legend()
plt.grid(True)
plt.show()

# 2) RMSE and MAE on validation
plt.figure(figsize=(8, 5))
plt.plot(epochs, history["val_rmse"], label="Val RMSE")
plt.plot(epochs, history["val_mae"], label="Val MAE")
plt.xlabel("Epoch")
plt.ylabel("Error")
plt.title("RMSE and MAE in Validation across epochs")
plt.legend()
plt.grid(True)
plt.show()

# 3) Pearson correlation and R² in validation
fig, ax1 = plt.subplots(figsize=(8, 5))

color1 = "tab:blue"
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Pearson (r)", color=color1)
ax1.plot(epochs, history["val_pearson"], marker="o", color=color1, label="Pearson (r)")
ax1.tick_params(axis="y", labelcolor=color1)
ax1.grid(True)

ax2 = ax1.twinx() 
color2 = "tab:red"
ax2.set_ylabel("R²", color=color2)
ax2.plot(epochs, history["val_r2"], marker="s", color=color2, label="R²")
ax2.tick_params(axis="y", labelcolor=color2)

plt.title("Pearson correlation and R² in validation.")
fig.tight_layout()
plt.show()


model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        out = model(data)
        all_preds.append(out.detach().cpu().numpy())
        all_labels.append(data.y.detach().cpu().numpy())

all_preds = np.concatenate(all_preds, axis=0).squeeze()
all_labels = np.concatenate(all_labels, axis=0).squeeze()

plt.figure(figsize=(6, 6))
plt.scatter(all_labels, all_preds, alpha=0.5)
min_val = min(all_labels.min(), all_preds.min())
max_val = max(all_labels.max(), all_preds.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--", label="Ideal line")
plt.xlabel("Real fidelity value")
plt.ylabel("Predicted Fidelity value")
plt.title("Prediction vs. Real in the Test Set")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Example of prediction in a single circuit.
print("\n--- xample of prediction ---")
model.eval()
sample_circuit = test_dataset[0].to(device)
predicted_pst = model(sample_circuit)
real_pst = sample_circuit.y

print(f"Test circuit:")
print(f"   - Real Fidelity(PST):    {real_pst:.6f}")
print(f"   - Predicted Fidelity (PST): {predicted_pst.item():.6f}")

### Usage Summary for Others

    Model Goal: This script trains the Graph Transformer to predict circuit fidelity (PST) between 0 and 1.

    Key Results: For replication (E1), expect R2≥0.93. For the 15–18 qubit regime (E3.2), the target R2 is approximately 0.796.

    Output: The code generates loss curves and a scatter plot of Predicted vs. Real PST. Ideally, points should cluster around the red dashed line ("Ideal line").